# log
1) customer id have 4 null values, for this projects this row will be delete.null in first and last name will be let it as it is ,while others will be replace by unknwon
2) remove white space in string type column
3) rename data in maritial and gender to be more reding friendly
4) rename each column 


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim,col

In [0]:
df=spark.table("workspace.bronze.crm_cust_info")
df.display()

In [0]:
#transformation

#finding null

null_counts=df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns])

# drop null
df=df.na.drop(subset=["cst_id"])

#trim string column
df=df.withColumn("cst_key",trim(col("cst_key")))
df=df.withColumn("cst_firstname",trim(col("cst_firstname")))
df=df.withColumn("cst_lastname",trim(col("cst_lastname")))

#normalization
df=(
    df.withColumn("cst_marital_status", F.when(F.upper(F.col("cst_marital_status")) == "M", "Married").
                  when(F.upper(F.col("cst_marital_status")) == "S", "Single").
                  otherwise("Unknown"))
    .withColumn("cst_gndr", F.when(F.upper(F.col("cst_gndr")) == "M", "Male").
                  when(F.upper(F.col("cst_gndr")) == "F","Female").
                  otherwise("Unknown"))
)

# renaming
rename_map = {"cst_id": "customer_id" ,"cst_key":"customer_number","cst_firstname":"first_name","cst_lastname":"last_name","cst_marital_status":"marital_status","cst_gndr":"gender",
              "cst_create_date":"create_date"}

df=df.withColumnsRenamed(rename_map)




In [0]:
# sanity check
df.display()

In [0]:
## writing to silver
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_cust_info")

In [0]:
%sql
SELECT *
FROM workspace.silver.crm_cust_info